<a href="https://colab.research.google.com/github/xyt556/I-GUIDE-GeoAI-Education/blob/main/notebooks/04-image-recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 图像识别

## 简介

图像识别（也称为图像分类或场景分类）为整个图像分配一个单一的分类标签。给定一个卫星或航空影像瓦片，分类器将其标记为“森林”、“住宅区”、“工业区”或类似。与分割不同，分割会标记每个像素，而分类则将整个瓦片视为一个单元。

这种瓦片级别的分类是地理空间AI的基础，支持土地利用测绘、城市扩张跟踪、环境监测和灾后评估。深度学习已经用卷积神经网络和视觉Transformer取代了手工设计的特征，这些网络直接从数据中学习判别特征。

在ImageNet上预训练的模型可以很好地迁移到遥感图像，因为边缘、纹理和颜色梯度等低级特征普遍有用。在本教程中，您将使用`geoai`包和[timm](https://github.com/huggingface/pytorch-image-models)库在EuroSAT卫星图像数据集上训练分类器。

## 学习目标

完成本教程后，您将能够：

- 解释图像识别与目标检测和语义分割的区别
- 描述卷积神经网络如何提取特征进行分类
- 比较流行的分类架构，包括ResNet、EfficientNet和Vision Transformers
- 准备ImageFolder风格的数据集以训练分类器
- 使用`geoai`高级API训练图像分类器
- 使用准确率、精确率、召回率、F1分数和混淆矩阵评估分类结果
- 将ImageNet预训练权重应用于遥感数据进行迁移学习
- 在相同数据集上比较多种架构

## 理解图像分类

### 从像素到标签

图像分类器接收一个固定大小的图像（例如，64 x 64像素，3个颜色通道），并生成预定义类别的概率分布，选择最高概率的类别作为预测。深度学习通过连续学习的卷积滤波器层直接从数据中学习特征，从而消除了手动特征工程。

### 卷积神经网络如何工作

卷积神经网络（CNN）通过一系列卷积层处理图像，这些卷积层检测越来越复杂的模式——从早期层中的边缘和梯度，到更深层中的屋顶、树冠和水体。特征提取后，全连接层将这些特征映射到类概率。

关键创新是权重共享：相同的滤波器应用于所有空间位置，因此在图像某一部分检测到的特征在任何地方都能被识别。这使得CNN既参数高效又对空间位移具有容忍度。

### 迁移学习

从头开始训练一个深度CNN需要数百万张标记图像，这在遥感任务中很少可用。迁移学习从在ImageNet（120万张照片，涵盖1000个类别）上预训练的模型开始，并在目标数据集上进行微调，所需标记样本数量大大减少，收敛速度更快。

两种常见的策略是：

- **完全微调**：在新的数据集上训练期间更新所有模型参数。这赋予模型最大的适应灵活性，但需要更多数据以避免过拟合。
- **骨干冻结**：预训练的卷积层被冻结（它们的权重不更新），只训练最终的分类头。这更快，并且在目标数据集较小的情况下效果很好，但限制了模型适应其特征表示的能力。

## 分类架构

### ResNet

残差网络（ResNets）引入了跳跃连接，允许梯度直接流过网络，从而实现更深层次的架构。每个块只学习输入和输出之间的残差差异，使优化更容易并避免梯度消失。

具有ImageNet预训练的ResNet-50是遥感分类的绝佳默认选择，以高效的训练提供强大的准确性。

### EfficientNet

EfficientNet使用复合缩放来统一缩放网络宽度、深度和分辨率，以更少的参数实现更高的准确性。该系列从B0（4.0M参数）到B7（63M参数）不等，提供了平滑的准确性-成本权衡。

对于卫星图像分类，EfficientNet-B0通常足够，并且比ResNet-50训练更快。

### Vision Transformers

Vision Transformers（ViTs）将图像分成固定大小的补丁，将其嵌入为向量，并使用Transformer自注意力进行处理。自注意力直接捕获长程依赖关系，但需要更大的训练数据集，并且计算成本与补丁数量呈二次方关系。

预训练的ViT模型在足够大的数据集上微调时，对于遥感可能非常有效。

### ConvNeXt

ConvNeXt将Transformer设计理念（更大的核、层归一化）融入卷积架构，在保持CNN效率的同时，达到或超越ViT的准确性。

### 选择架构

对于大多数地理空间任务，**ResNet-50**是一个可靠的起点。**EfficientNet-B0**以更少的参数提供可比的准确性。对于更大的数据集，**ConvNeXt**或**ViT**可能提供额外的增益。`geoai`包通过[timm](https://github.com/huggingface/pytorch-image-models)支持1000多种架构，通过简单的字符串参数指定。

## 准备分类数据

### ImageFolder结构

图像分类数据集的标准格式是ImageFolder布局：一个根目录，每个类别一个子目录，每个子目录包含该类别的图像。

```
dataset/
├── AnnualCrop/
│   ├── image_001.jpg
│   ├── image_002.jpg
│   └── ...
├── Forest/
│   ├── image_001.jpg
│   └── ...
├── Residential/
│   ├── image_001.jpg
│   └── ...
└── ...
```

目录名成为类别标签。`geoai.recognize`模块会自动扫描此结构。

### EuroSAT数据集

[EuroSAT](https://github.com/phelber/eurosat)数据集是Sentinel-2图像土地利用/土地覆盖分类中广泛使用的基准。RGB子集包含27,000个图像块（64 x 64像素），分为10个类别：

- `AnnualCrop`
- `Forest`
- `HerbaceousVegetation`
- `Highway`
- `Industrial`
- `Pasture`
- `PermanentCrop`
- `Residential`
- `River`
- `SeaLake`

每个类别有2,000到3,000张图像，分辨率为10米，覆盖34个欧洲国家。

## 安装

取消注释以下行以安装所需的包。

In [ ]:
# %pip install -U "geoai-py[extra]"

## 导入库

`geoai.recognize`模块为完整的分类管道提供了高级API。本教程中使用的主要函数是：

- `load_image_dataset`：扫描ImageFolder目录并返回元数据（类名、图像路径、标签映射）
- `train_image_classifier`：处理数据集分割、模型构建、训练循环和保存检查点
- `evaluate_classifier`：在数据集分割上运行推理并返回准确率、每个类别的分类报告和混淆矩阵
- `plot_training_history`：读取保存的训练日志并绘制每个epoch的损失和准确率曲线
- `plot_confusion_matrix`：渲染`evaluate_classifier`返回的混淆矩阵
- `predict_images`：对图像文件路径列表运行推理并返回预测的类名和置信度分数
- `plot_predictions`：显示带有预测标签和真实标签的图像网格
- `push_classifier_to_hub`：将训练好的分类器检查点上传到Hugging Face Hub仓库
- `predict_images_from_hub`：从Hub下载分类器并在无需本地重新训练的情况下运行推理

In [ ]:
import os
from geoai.utils import download_file
from geoai.recognize import (
    load_image_dataset,
    train_image_classifier,
    predict_images,
    evaluate_classifier,
    plot_training_history,
    plot_confusion_matrix,
    plot_predictions,
    push_classifier_to_hub,
    predict_images_from_hub,
)

## 下载EuroSAT RGB数据集

使用`download_file`实用程序下载并解压EuroSAT数据集。

In [ ]:
url = "https://data.source.coop/opengeos/geoai/EuroSAT-RGB.zip"
data_dir = download_file(url)

In [ ]:
print(f"Dataset directory: {data_dir}")
print(f"Files: {sorted(os.listdir(data_dir))}")

## 探索数据集

`load_image_dataset`函数扫描ImageFolder目录并返回一个包含四个键的字典：

- `class_names`：从子目录名派生的类名排序列表
- `image_paths`：每个图像文件的绝对路径列表
- `labels`：每个图像的整数标签（`class_names`的索引）
- `class_to_idx`：从类名字符串到整数索引的映射

In [ ]:
dataset_info = load_image_dataset(data_dir)
print(f"Classes ({len(dataset_info['class_names'])}): {dataset_info['class_names']}")
print(f"Total images: {len(dataset_info['image_paths'])}")

```text
在10个类别中找到27000张图像
  AnnualCrop: 3000
  Forest: 3000
  HerbaceousVegetation: 3000
  Highway: 2500
  Industrial: 2500
  Pasture: 2000
  PermanentCrop: 2500
  Residential: 3000
  River: 2500
  SeaLake: 3000
类别 (10): ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
总图像: 27000
```

可视化每个类别的一张代表性图像，以确认下载成功并了解每个土地覆盖类别是什么样子的。

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

class_names = dataset_info["class_names"]
image_paths = dataset_info["image_paths"]
labels = dataset_info["labels"]
class_to_idx = dataset_info["class_to_idx"]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for idx, class_name in enumerate(class_names):
    ax = axes[idx // 5, idx % 5]
    # Find first image of this class
    img_idx = labels.index(class_to_idx[class_name])
    img = Image.open(image_paths[img_idx])
    ax.imshow(img)
    ax.set_title(class_name, fontsize=12)
    ax.axis("off")

plt.suptitle("Sample Image from Each Class", fontsize=14)
plt.tight_layout()
plt.show()

## 训练ResNet-50分类器

`train_image_classifier`函数处理完整的管道：扫描ImageFolder目录，执行分层70/15/15分割，构建模型，运行训练，并保存最佳检查点。主要参数包括：

- `model_name`：任何`timm`架构标识符（例如，`"resnet50"`，`"efficientnet_b0"`）。请浏览[timm模型索引](https://huggingface.co/timm)以获取完整列表。
- `pretrained=True`：使用ImageNet权重进行初始化以进行迁移学习
- `num_epochs`：训练集上的迭代次数
- `batch_size`和`learning_rate`：优化超参数
- `image_size`和`in_channels`：必须与数据集匹配（EuroSAT RGB为64 x 64像素，3个通道）
- `output_dir`：保存训练日志和最佳检查点的位置
- `seed`：固定随机分割和初始化以实现可重现性

该函数返回一个字典，其中包含训练好的`model`、`class_names`、`checkpoint_path`和预分割的数据集。

In [ ]:
result = train_image_classifier(
    data_dir=data_dir,
    model_name="resnet50",
    num_epochs=5,
    batch_size=32,
    learning_rate=1e-3,
    image_size=64,
    in_channels=3,
    pretrained=True,
    output_dir="image_recognition_output/resnet50",
    num_workers=4,
    seed=42,
)

## 绘制训练历史

绘制训练和验证损失/准确率随epoch的变化曲线，揭示学习进度和潜在的过拟合。如果验证损失上升而训练损失下降，则模型正在过拟合。

In [ ]:
fig = plot_training_history("image_recognition_output/resnet50/models")
plt.show()

## 在测试集上评估

`evaluate_classifier`函数在保留的测试集上运行训练好的模型，并打印包含四个指标的每个类别的分类报告：

- **精确率（Precision）**：在所有预测为X类的瓦片中，有多少比例实际属于X类？
- **召回率（Recall）**：在所有实际属于X类的瓦片中，模型正确识别了多少比例？
- **F1分数（F1-score）**：精确率和召回率的调和平均值。
- **支持度（Support）**：每个类别中的测试图像数量。

该函数返回一个包含整体`accuracy`、`classification_report`字符串和`confusion_matrix`数组的字典。

In [ ]:
eval_result = evaluate_classifier(
    model=result["model"],
    dataset=result["test_dataset"],
    class_names=result["class_names"],
)

## 绘制混淆矩阵

混淆矩阵显示了真实类别（行）和预测类别（列）的所有组合。对角线单元格是正确预测；非对角线单元格是错误。我们绘制原始计数和归一化版本（将每行除以其总和），以揭示每个类别的错误率。

In [ ]:
fig = plot_confusion_matrix(
    eval_result["confusion_matrix"],
    result["class_names"],
)
plt.show()

In [ ]:
fig = plot_confusion_matrix(
    eval_result["confusion_matrix"],
    result["class_names"],
    normalize=True,
)
plt.show()

## 可视化预测

显示随机测试图像上的预测有助于检查成功和失败。绿色标题表示正确预测；红色标题表示错误分类。

In [ ]:
import random

test_dataset = result["test_dataset"]
test_paths = test_dataset.image_paths
test_labels = test_dataset.labels
n_samples = min(10, len(test_paths))

rng = random.Random(42)
sample_indices = rng.sample(range(len(test_paths)), k=n_samples)
sample_paths = [test_paths[i] for i in sample_indices]
sample_labels = [test_labels[i] for i in sample_indices]

pred_result = predict_images(
    model=result["model"],
    image_paths=sample_paths,
    class_names=result["class_names"],
    image_size=64,
    in_channels=3,
)

fig = plot_predictions(
    image_paths=sample_paths,
    predictions=pred_result["predictions"],
    true_labels=sample_labels,
    class_names=result["class_names"],
    probabilities=pred_result["probabilities"],
)
plt.show()

## 训练EfficientNet-B0分类器

EfficientNet-B0通过使用复合缩放，以大约五分之一的参数（4M vs. 25M）实现了与ResNet-50相当的准确率。所有超参数保持不变以进行公平比较；只改变了`model_name`和`output_dir`。

In [ ]:
result_effnet = train_image_classifier(
    data_dir=data_dir,
    model_name="efficientnet_b0",
    num_epochs=5,
    batch_size=32,
    learning_rate=1e-3,
    image_size=64,
    in_channels=3,
    pretrained=True,
    output_dir="image_recognition_output/efficientnet_b0",
    num_workers=4,
    seed=42,
)

比较两个模型的训练曲线，揭示收敛速度和过拟合行为。

In [ ]:
fig = plot_training_history("image_recognition_output/efficientnet_b0/models")
plt.show()

In [ ]:
eval_effnet = evaluate_classifier(
    model=result_effnet["model"],
    dataset=result_effnet["test_dataset"],
    class_names=result_effnet["class_names"],
)

将EfficientNet-B0的归一化混淆矩阵与上面ResNet-50的混淆矩阵进行比较，以了解架构选择如何影响每个类别的性能。

In [ ]:
fig = plot_confusion_matrix(
    eval_effnet["confusion_matrix"],
    result_effnet["class_names"],
    normalize=True,
)
plt.show()

## 比较结果

总体测试准确率提供了快速摘要，但如果模型更小、训练更快或在边缘硬件上运行更好，则即使准确率稍低，也可能更受青睐。

In [ ]:
print(f"ResNet50 accuracy:        {eval_result['accuracy']:.4f}")
print(f"EfficientNet-B0 accuracy: {eval_effnet['accuracy']:.4f}")

| 模型           | 参数       | 典型EuroSAT准确率 | 相对速度 |
| --------------- | ---------- | -------------------- | -------- |
| ResNet-50       | ~25M       | ~93–97%              | 基准     |
| EfficientNet-B0 | ~4.0M      | ~93–97%              | 更快     |

两种架构在微调后表现相似。EfficientNet-B0是大规模部署的强默认选择，而ResNet-50提供了最多的已发布基准。

## 发布和重用模型

通过[Hugging Face Hub](https://huggingface.co/docs/hub)共享训练好的模型，可以让协作者立即运行推理，而无需原始训练数据或计算资源。

### Hugging Face身份验证

要推送模型，您需要一个免费的Hugging Face帐户和写权限令牌。`notebook_login`函数将令牌存储在本地。

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

### 将训练好的模型推送到Hub

`push_classifier_to_hub`函数将最佳检查点作为`model.pth`和`config.json`上传到Hub仓库。将`"your-username"`替换为您的Hugging Face用户名。

In [ ]:
repo_url = push_classifier_to_hub(
    model_path=result["checkpoint_path"],
    repo_id="your-username/eurosat-resnet50",
    model_name="resnet50",
    num_classes=len(result["class_names"]),
    in_channels=3,
    class_names=result["class_names"],
    commit_message="EuroSAT ResNet-50 classifier trained for 5 epochs",
)
print(repo_url)

### 从Hub运行推理

`predict_images_from_hub`函数从Hub下载模型和配置，并直接运行推理。不需要本地检查点或类名列表。

In [ ]:
n_samples = 10
hub_result = predict_images_from_hub(
    image_paths=test_paths[:n_samples],
    repo_id="your-username/eurosat-resnet50",
    image_size=64,
)

fig = plot_predictions(
    image_paths=test_paths[:n_samples],
    predictions=hub_result["predictions"],
    true_labels=test_labels[:n_samples],
    class_names=hub_result["class_names"],
    probabilities=hub_result["probabilities"],
)
plt.show()

## 关键要点

1. 图像识别为整个图像瓦片分配一个单一的类标签，使其成为地理空间AI中最简单的视觉理解任务。

2. 使用预训练的ImageNet模型进行迁移学习大大减少了所需的标记数据，因为低级视觉特征是领域无关的。

3. ImageFolder格式（每个类别一个子目录）是标准数据集布局，由`geoai.recognize`自动扫描。

4. ResNet-50是一个可靠的默认选择；EfficientNet以更少的参数提供可比的准确性；ViT和ConvNeXt适用于更大的数据集。

5. 混淆矩阵揭示了聚合准确率指标隐藏的类别级错误。

6. `geoai`高级API简化了从训练到评估的整个管道，只需最少的样板代码。